# **BFP_PE Synthesis Flow — TSMC28 HPC+**

Adapted from CA_RISCV.ipynb. This notebook drives synthesis of the Chisel-generated `BFP_PE` design using Synopsys Design Compiler with the TSMC28 HPC+ PDK.

**Before running on the student server, ensure:**
```bash
export PDK_ROOT=/users/micas/micasusr/design/tsmc28hpcplus
export PDK=tsmc28hpcplus
```

### **Helper: write-template magic**

In [ ]:
import os, time
import matplotlib.pyplot as plt
import pandas as pd
from pandas.core.frame import DataFrame
pd.set_option('precision', 3)
pd.set_option('display.width', 200)
pd.set_option('max_colwidth', 90)

In [ ]:
from IPython.core.magic import register_line_cell_magic

@register_line_cell_magic
def writetemplate(line, cell):
    with open(line, 'w') as f:
        f.write(cell.format(**globals()))

### **Project-specific Environment Variables**

- `CONFIG` selects one of the generated configurations under `test_fused_dot/`.
- `RTL_PATH` is automatically derived from `CONFIG`.
- `PDK_ROOT` must match the student-server path for TSMC28 (read from env var, falls back to default).
- Adjust `CLOCK_PERIOD` (in ns) for your target frequency (default: 1 GHz → 1 ns).
- Library discovery is done **inside the TCL script** at runtime on the server, so no file-system access to the PDK is needed here.

In [ ]:
# ── Configuration selector ────────────────────────────────────────────────────
CONFIG = "INT8_E2M1_UE8M0_vec1"
# CONFIG = "INT8_E2M1_UE8M0_vec2"
# CONFIG = "INT8_E2M1_UE8M0_vec4"
# CONFIG = "INT8_E2M1_UE8M0_vec8"
# CONFIG = "INT8_E2M1_UE8M0_vec16"
# CONFIG = "INT8_E2M1_UE8M0_vec32"

DESIGN_NAME  = "BFP_PE"
CLOCK_PERIOD = 1.0   # nanoseconds  (1 ns = 1 GHz)
CLOCK_PORT   = "clock"

# ── Paths ─────────────────────────────────────────────────────────────────────
# Absolute path to the repo root — works both in Docker and on the server
# as long as the repo is cloned to the same location.
REPO_ROOT = os.path.abspath(
    os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "..", "..")
)
RTL_PATH = os.path.join(
    REPO_ROOT, "hw", "chisel_acc", "generated", "test_fused_dot",
    CONFIG, f"{DESIGN_NAME}.sv"
)

# ── PDK (TSMC28 HPC+) ─────────────────────────────────────────────────────────
# Read from environment variables set on the student server.
# No filesystem access to PDK_ROOT is done here; library discovery
# happens inside the generated TCL script at synthesis time.
PDK_ROOT = os.environ.get("PDK_ROOT", "/users/micas/micasusr/design/tsmc28hpcplus")
PDK      = os.environ.get("PDK",      "tsmc28hpcplus")

# Standard-cell library root — string construction only, no I/O
STD_CELL_LIB_ROOT = f"{PDK_ROOT}/standard_cell_libraries"

# ── Run tag ───────────────────────────────────────────────────────────────────
TAG      = "{}_{}".format(time.strftime("%y%m%d-%H%M%S", time.localtime()), CONFIG)
CORE_NUM = os.popen("nproc").read().strip()

print(f"CONFIG           : {CONFIG}")
print(f"RTL_PATH         : {RTL_PATH}")
print(f"PDK_ROOT         : {PDK_ROOT}")
print(f"STD_CELL_LIB_ROOT: {STD_CELL_LIB_ROOT}")
print(f"TAG              : {TAG}")

## **Synthesis — Step 1: Generate the DC synthesis script**

Writes `synth.tcl` which is fed to `dc_shell`.

The `find` command for `.db` liberty files runs **inside TCL on the server**, so it resolves correctly against the real TSMC28 installation.

In [ ]:
%%writetemplate synth.tcl

# ── Synopsys Design Compiler synthesis script ─────────────────────────────────
# Generated by BFP_PE_synth.ipynb
# CONFIG : {CONFIG}
# RTL    : {RTL_PATH}

set DESIGN_NAME       "{DESIGN_NAME}"
set RTL_FILE          "{RTL_PATH}"
set STD_CELL_LIB_ROOT "{STD_CELL_LIB_ROOT}"
set TAG               "{TAG}"

# ── Technology libraries (discovered at runtime on the server) ────────────────
# Find all .db files under the standard-cell library root.
set _lib_file_str [exec find $STD_CELL_LIB_ROOT -name "*.db" -maxdepth 4]
set target_library [split [string trim $_lib_file_str] "\n"]
set link_library   [concat "*" $target_library]
set search_path    [concat $search_path [list $STD_CELL_LIB_ROOT]]

puts "Using [llength $target_library] library file(s):"
foreach lib $target_library {{ puts "  $lib" }}

# ── Output directories ────────────────────────────────────────────────────────
set RESULTS_DIR "./runs/$TAG/results"
set REPORTS_DIR "./runs/$TAG/reports"
file mkdir $RESULTS_DIR
file mkdir $REPORTS_DIR

# ── Read & elaborate ─────────────────────────────────────────────────────────
analyze -format sverilog $RTL_FILE
elaborate $DESIGN_NAME
current_design $DESIGN_NAME
link

# ── Constraints ───────────────────────────────────────────────────────────────
create_clock -period {CLOCK_PERIOD} -name clk [get_ports {CLOCK_PORT}]
set_input_delay  0 -clock clk [all_inputs]
set_output_delay 0 -clock clk [all_outputs]

# ── Compile ───────────────────────────────────────────────────────────────────
compile_ultra

# ── Reports ───────────────────────────────────────────────────────────────────
report_area    -hierarchy > $REPORTS_DIR/area.rpt
report_timing  -max_paths 10 > $REPORTS_DIR/timing.rpt
report_power   > $REPORTS_DIR/power.rpt
report_cell    > $REPORTS_DIR/cells.rpt

# ── Write netlist ─────────────────────────────────────────────────────────────
write -format verilog -hierarchy -output $RESULTS_DIR/${DESIGN_NAME}_netlist.v
write_sdf $RESULTS_DIR/${DESIGN_NAME}.sdf

puts "\n[SUCCESS] Synthesis complete: $TAG"
exit


## **Synthesis — Step 2: Run dc_shell**

Run this cell on the student server where `dc_shell` and the TSMC28 PDK are available.

In [ ]:
os.makedirs("runs", exist_ok=True)
!dc_shell -f synth.tcl | tee runs/{TAG}_dc.log

## **Analysis — Area Report**

In [ ]:
LOG_PATH = f"./runs/{TAG}/reports/area.rpt"
_stdout = os.popen(f"grep -E 'Hierarchical|Number of|Total cell area' {LOG_PATH}").read()
print(_stdout)

## **Analysis — Timing Report**

In [ ]:
with open(f"./runs/{TAG}/reports/timing.rpt") as f:
    print(f.read())

## **Batch synthesis over all configurations**

Run all six `vec` configurations and collect area + WNS in a summary table.

In [ ]:
CONFIGS = [
    "INT8_E2M1_UE8M0_vec1",
    "INT8_E2M1_UE8M0_vec2",
    "INT8_E2M1_UE8M0_vec4",
    "INT8_E2M1_UE8M0_vec8",
    "INT8_E2M1_UE8M0_vec16",
    "INT8_E2M1_UE8M0_vec32",
]

os.makedirs("runs", exist_ok=True)
results = []

for cfg in CONFIGS:
    cfg_tag = "{}_{}".format(time.strftime("%y%m%d-%H%M%S", time.localtime()), cfg)
    rtl = os.path.join(
        REPO_ROOT, "hw", "chisel_acc", "generated", "test_fused_dot",
        cfg, f"{DESIGN_NAME}.sv"
    )

    # Write per-config TCL — library discovery happens inside TCL at runtime
    tcl_content = f"""set DESIGN_NAME       "{DESIGN_NAME}"
set RTL_FILE          "{rtl}"
set STD_CELL_LIB_ROOT "{STD_CELL_LIB_ROOT}"
set TAG               "{cfg_tag}"
set _lib_file_str [exec find $STD_CELL_LIB_ROOT -name "*.db" -maxdepth 4]
set target_library [split [string trim $_lib_file_str] "\\n"]
set link_library   [concat "*" $target_library]
set search_path    [concat $search_path [list $STD_CELL_LIB_ROOT]]
file mkdir ./runs/$TAG/results
file mkdir ./runs/$TAG/reports
analyze -format sverilog $RTL_FILE
elaborate $DESIGN_NAME
current_design $DESIGN_NAME
link
create_clock -period {CLOCK_PERIOD} -name clk [get_ports {CLOCK_PORT}]
set_input_delay  0 -clock clk [all_inputs]
set_output_delay 0 -clock clk [all_outputs]
compile_ultra
report_area    -hierarchy > ./runs/$TAG/reports/area.rpt
report_timing  -max_paths 10 > ./runs/$TAG/reports/timing.rpt
report_power   > ./runs/$TAG/reports/power.rpt
write -format verilog -hierarchy -output ./runs/$TAG/results/{DESIGN_NAME}_netlist.v
exit
"""
    tcl_file = f"synth_{cfg}.tcl"
    with open(tcl_file, "w") as f:
        f.write(tcl_content)

    print(f"Running {cfg} ...")
    os.system(f"dc_shell -f {tcl_file} > runs/{cfg_tag}_dc.log 2>&1")

    area_line = os.popen(f"grep 'Total cell area' runs/{cfg_tag}/reports/area.rpt").read().strip()
    area_val  = float(area_line.split()[-1]) if area_line else float('nan')

    wns_line  = os.popen(f"grep 'slack' runs/{cfg_tag}/reports/timing.rpt | tail -1").read().strip()
    wns_val   = float(wns_line.split()[-1]) if wns_line else float('nan')

    results.append({"Config": cfg, "Area (um^2)": area_val, "WNS (ns)": wns_val})
    print(f"  -> area={area_val:.2f} um^2, WNS={wns_val:.3f} ns")

df = DataFrame(results)
print("\n", df.to_string(index=False))